In [13]:
# CELL 1: CÀI ĐẶT THƯ VIỆN CẦN THIẾT
import subprocess
import sys

print("="*70)
print("📦 CÀI ĐẶT THƯ VIỆN")
print("="*70)

# Cài đặt underthesea
subprocess.check_call([sys.executable, "-m", "pip", "install", "underthesea", "-q"])

# Cài đặt gensim (nếu chưa có)
subprocess.check_call([sys.executable, "-m", "pip", "install", "gensim", "-q"])

# Cài đặt wordcloud
subprocess.check_call([sys.executable, "-m", "pip", "install", "wordcloud", "-q"])

print("✅ Đã cài đặt xong các thư viện!")

📦 CÀI ĐẶT THƯ VIỆN
✅ Đã cài đặt xong các thư viện!


In [14]:
# CELL 1: KHỞI TẠO - KHÔNG CẦN PATCH
import sys
import os
import warnings
import datetime

# ============ TẮT CẢNH BÁO ============
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# ============ IMPORT THƯ VIỆN ============
import pandas as pd
import numpy as np
import re
import glob
import time
import random
import json
import matplotlib.pyplot as plt
from collections import Counter

# NLP - TIẾNG VIỆT
from underthesea import word_tokenize

# GENSIM LDA
import gensim
from gensim import corpora, models
from gensim.models import CoherenceModel

# VISUALIZATION
from wordcloud import WordCloud

print("="*70)
print("✅ KHỞI TẠO THÀNH CÔNG")
print("="*70)
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"📦 Pandas: {pd.__version__}")
print(f"📦 Gensim: {gensim.__version__}")
print(f"📦 Underthesea: {word_tokenize.__module__}")

print("\n" + "="*70)
print("⚠️ LƯU Ý QUAN TRỌNG:")
print("   Cảnh báo DeprecationWarning từ Jupyter Client KHÔNG ẢNH HƯỞNG")
print("   Code vẫn chạy bình thường và training LDA vẫn hoạt động!")
print("="*70)

✅ KHỞI TẠO THÀNH CÔNG
🐍 Python: 3.12.12
📦 Pandas: 2.3.3
📦 Gensim: 4.4.0
📦 Underthesea: underthesea.pipeline.word_tokenize

⚠️ LƯU Ý QUAN TRỌNG:
   Cảnh báo DeprecationWarning từ Jupyter Client KHÔNG ẢNH HƯỞNG
   Code vẫn chạy bình thường và training LDA vẫn hoạt động!


In [15]:
# CELL 3: ĐỌC DỮ LIỆU - TĂNG LÊN 100,000+ DOCUMENTS
print("="*70)
print("📖 ĐỌC DỮ LIỆU - TỐI ĐA CHO RAM 30GB")
print("="*70)

import os
import pandas as pd
import glob
import gc

# Tìm tất cả file CSV
csv_files = []
for root, dirs, files in os.walk("/kaggle/input/"):
    for file in files:
        if file.endswith('.csv') and 'sample' not in file:
            csv_files.append(os.path.join(root, file))

print(f"✅ Tìm thấy {len(csv_files)} file CSV")

if len(csv_files) == 0:
    print("❌ Không tìm thấy file CSV!")
    import sys
    sys.exit(0)

# ============ CẤU HÌNH ============
NUM_FILES = 8           # Đọc 8 file
ROWS_PER_FILE = 20000   # Mỗi file 20,000 dòng
# Tổng: 8 × 20,000 = 160,000 documents

print(f"\n📊 CẤU HÌNH:")
print(f"   - Số file: {NUM_FILES}")
print(f"   - Dòng/file: {ROWS_PER_FILE:,}")
print(f"   - Tổng dòng dự kiến: {NUM_FILES * ROWS_PER_FILE:,}")

# Đọc dữ liệu
all_documents = []
total_rows = 0

for i in range(min(NUM_FILES, len(csv_files))):
    file = csv_files[i]
    print(f"\n[{i+1}/{NUM_FILES}] {os.path.basename(file)}")
    
    try:
        # Đọc theo chunk để kiểm soát RAM
        chunks = []
        for chunk in pd.read_csv(file, chunksize=5000, nrows=ROWS_PER_FILE):
            text_col = chunk.columns[0]
            docs = chunk[text_col].fillna('').astype(str).tolist()
            chunks.extend(docs)
            total_rows += len(docs)
        
        all_documents.extend(chunks)
        print(f"   ✅ Đã đọc {len(chunks):,} docs | Tổng: {total_rows:,}")
        
        # Giải phóng bộ nhớ tạm
        del chunks
        gc.collect()
        
    except Exception as e:
        print(f"   ❌ Lỗi: {e}")

print(f"\n✅ TỔNG CỘNG: {total_rows:,} documents")
print(f"💾 RAM ước tính hiện tại: ~{total_rows * 0.0001:.1f} GB")

# Giải phóng
gc.collect()

📖 ĐỌC DỮ LIỆU - TỐI ĐA CHO RAM 30GB
✅ Tìm thấy 22 file CSV

📊 CẤU HÌNH:
   - Số file: 8
   - Dòng/file: 20,000
   - Tổng dòng dự kiến: 160,000

[1/8] vietnamese_texts_part_0020.csv
   ✅ Đã đọc 20,000 docs | Tổng: 20,000

[2/8] vietnamese_texts_part_0015.csv
   ✅ Đã đọc 20,000 docs | Tổng: 40,000

[3/8] vietnamese_texts_part_0006.csv
   ✅ Đã đọc 20,000 docs | Tổng: 60,000

[4/8] vietnamese_texts_part_0014.csv
   ✅ Đã đọc 20,000 docs | Tổng: 80,000

[5/8] vietnamese_texts_part_0012.csv
   ✅ Đã đọc 20,000 docs | Tổng: 100,000

[6/8] vietnamese_texts_part_0004.csv
   ✅ Đã đọc 20,000 docs | Tổng: 120,000

[7/8] vietnamese_texts_part_0009.csv
   ✅ Đã đọc 20,000 docs | Tổng: 140,000

[8/8] vietnamese_texts_part_0002.csv
   ✅ Đã đọc 20,000 docs | Tổng: 160,000

✅ TỔNG CỘNG: 160,000 documents
💾 RAM ước tính hiện tại: ~16.0 GB


0

In [16]:
# CELL 4: LẤY MẪU - TĂNG LÊN 100,000 DOCUMENTS
print("="*70)
print("📊 LẤY MẪU DỮ LIỆU - 100,000 DOCUMENTS")
print("="*70)

# ============ CẤU HÌNH ============
# BẠN CÓ THỂ CHỌN 1 TRONG 3 MỨC:

print("Chọn mức dữ liệu:")
print("1. 50,000 documents (nhanh, test)")
print("2. 100,000 documents (khuyến nghị)")
print("3. 150,000 documents (tối đa cho RAM 30GB)")

choice = input("\nNhập lựa chọn (1/2/3): ")

if choice == "1":
    SAMPLE_SIZE = 50000
    print(f"\n✅ Chọn: 50,000 documents")
elif choice == "2":
    SAMPLE_SIZE = 100000
    print(f"\n✅ Chọn: 100,000 documents")
elif choice == "3":
    SAMPLE_SIZE = 150000
    print(f"\n✅ Chọn: 150,000 documents")
else:
    SAMPLE_SIZE = 100000
    print(f"\n✅ Mặc định: 100,000 documents")

if total_rows > SAMPLE_SIZE:
    print(f"Lấy mẫu {SAMPLE_SIZE:,} / {total_rows:,} documents")
    random.seed(42)
    sample_docs = random.sample(all_documents, SAMPLE_SIZE)
else:
    print(f"Sử dụng toàn bộ {total_rows:,} documents")
    sample_docs = all_documents

print(f"✅ Sample size: {len(sample_docs):,}")
print(f"💾 RAM ước tính: ~{len(sample_docs) * 0.0001:.1f} GB")

# Giải phóng all_documents
del all_documents
gc.collect()

# Hiển thị mẫu
print("\n📄 MẪU VĂN BẢN (3 document đầu):")
for i in range(min(3, len(sample_docs))):
    text = sample_docs[i][:200]
    print(f"\n--- Document {i+1} ---")
    print(text + "..." if len(sample_docs[i]) > 200 else text)

📊 LẤY MẪU DỮ LIỆU - 100,000 DOCUMENTS
Chọn mức dữ liệu:
1. 50,000 documents (nhanh, test)
2. 100,000 documents (khuyến nghị)
3. 150,000 documents (tối đa cho RAM 30GB)



Nhập lựa chọn (1/2/3):  3



✅ Chọn: 150,000 documents
Lấy mẫu 150,000 / 160,000 documents
✅ Sample size: 150,000
💾 RAM ước tính: ~15.0 GB

📄 MẪU VĂN BẢN (3 document đầu):

--- Document 1 ---
Microsoft Office 2007 là phiên bạn dạng cách tân của phiên bản Office 2003 trước đó. Phiên bạn dạng Office 2007 được tăng cấp tương đối nhiều về tính năng cùng một số biến hóa về bối cảnh.
Bạn đang xe...

--- Document 2 ---
Kẻ Hạ xư­a nay vẫn đ­ược coi là vùng thắng tích bậc nhất đất Chi La - La Sơn - Đức Thọ với Tùng Lĩnh, La Giang, Mai Hồ, Ngưu Chử; là nơi đóng quân và có đền thờ hai danh t­ướng nghĩa quân Lam Sơn là L...

--- Document 3 ---
Đấm Phát Gục Luôn - Lực Đấm 534 Kg - Huyền Thoại Làng BOXING | Mike Tayson
Đấm Phát Gục Luôn – Lực Đấm 534 Kg – Huyền Thoại Làng BOXING | Mike Tayson
Đấm Ra Lực 534 Kg – Bàn Tay Của Mike Tayson Đã Khổ...


In [17]:
# CELL 5: STOPWORDS TIẾNG VIỆT
print("="*70)
print("🔤 STOPWORDS TIẾNG VIỆT")
print("="*70)

STOPWORDS = {
    'và', 'của', 'một', 'những', 'cho', 'với', 'các', 'đã', 'sẽ', 'có', 'còn',
    'được', 'không', 'này', 'nên', 'như', 'khi', 'vào', 'ra', 'thì', 'mà', 'rất',
    'để', 'cũng', 'nếu', 'vì', 'tại', 'hơn', 'nữa', 'lại', 'thấy', 'biết',
    'đó', 'là', 'từ', 'trong', 'ngoài', 'trên', 'dưới', 'trước', 'sau', 'nay',
    'năm', 'tháng', 'ngày', 'giờ', 'phút', 'giây', 'lúc', 'lên', 'xuống', 'đi',
    'lại', 'về', 'tới', 'đến', 'rồi', 'mới', 'cũ', 'đủ', 'nhiều', 'ít', 'quá',
    'lắm', 'đều', 'cả', 'đấy', 'ấy', 'đây', 'đó', 'nọ', 'kia', 'vậy', 'nên',
    'chỉ', 'nhưng', 'hoặc', 'cùng', 'giữa', 'bằng', 'vẫn', 'luôn', 'bị', 'đang',
    'làm', 'đã', 'sẽ', 'được', 'hãy', 'đừng', 'chớ', 'nhé', 'ạ', 'ơi', 'nhỉ',
    'ư', 'hả', 'à', 'á', 'ả', 'ã', 'ạ', 'ă', 'ắ', 'ằ', 'ẳ', 'ẵ', 'ặ',
    'â', 'ấ', 'ầ', 'ẩ', 'ẫ', 'ậ', 'ê', 'ế', 'ề', 'ể', 'ễ', 'ệ',
    'ô', 'ố', 'ồ', 'ổ', 'ỗ', 'ộ', 'ơ', 'ớ', 'ờ', 'ở', 'ỡ', 'ợ',
    'ư', 'ứ', 'ừ', 'ử', 'ữ', 'ự'
}

print(f"✅ Số stopwords: {len(STOPWORDS)}")
print(f"📝 Ví dụ 20 stopwords đầu tiên:")
for i, word in enumerate(list(STOPWORDS)[:20]):
    print(f"   {i+1:2d}. {word}")

🔤 STOPWORDS TIẾNG VIỆT
✅ Số stopwords: 130
📝 Ví dụ 20 stopwords đầu tiên:
    1. nữa
    2. như
    3. với
    4. ằ
    5. nếu
    6. ỗ
    7. ỡ
    8. về
    9. ớ
   10. đi
   11. ầ
   12. â
   13. còn
   14. có
   15. ệ
   16. ẫ
   17. tới
   18. ư
   19. ạ
   20. dưới


In [18]:
# CELL 6: HÀM TIỀN XỬ LÝ VĂN BẢN
print("="*70)
print("🔧 HÀM TIỀN XỬ LÝ VĂN BẢN TIẾNG VIỆT")
print("="*70)

def clean_text(text):
    """
    Làm sạch văn bản: lowercase, loại bỏ ký tự đặc biệt, số
    """
    if not isinstance(text, str):
        return ""
    
    # Chuyển về chữ thường
    text = text.lower()
    
    # Loại bỏ số và ký tự đặc biệt (chỉ giữ chữ cái tiếng Việt và khoảng trắng)
    text = re.sub(r'[^a-zA-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠàáâãèéêìíòóôõùúăđĩũơƯĂẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼỀỀỂẾưăạảấầẩẫậắằẳẵặẹẻẽềềểế\s]', ' ', text)
    
    # Xóa khoảng trắng thừa
    text = ' '.join(text.split())
    
    return text

def tokenize_vietnamese(text):
    """
    Tokenize văn bản tiếng Việt sử dụng underthesea
    """
    try:
        # word_tokenize trả về các từ đã được tách
        tokens = word_tokenize(text, format="text")
        return tokens.split()
    except Exception as e:
        # Fallback: tách theo khoảng trắng nếu lỗi
        print(f"   ⚠️ Tokenize fallback: {e}")
        return text.split()

def filter_tokens(tokens):
    """
    Lọc token: loại bỏ stopwords và từ quá ngắn
    """
    # Loại bỏ stopwords và từ có độ dài <= 2
    filtered = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return filtered

def preprocess(text):
    """
    Pipeline tiền xử lý hoàn chỉnh
    """
    # Bước 1: Làm sạch
    text = clean_text(text)
    
    # Bước 2: Tokenize
    tokens = tokenize_vietnamese(text)
    
    # Bước 3: Lọc
    tokens = filter_tokens(tokens)
    
    return tokens

# Test hàm với các ví dụ
print("\n🧪 TEST HÀM TIỀN XỬ LÝ:")
print("-"*50)

test_cases = [
    "Hôm nay là một ngày đẹp trời, tôi đi học và gặp nhiều bạn bè!",
    "Thời tiết hôm nay thật tuyệt vời, trời xanh mây trắng.",
    "iPhone 15 Pro Max vừa được Apple ra mắt với nhiều tính năng mới.",
    "Bộ trưởng Bộ Giáo dục và Đào tạo phát biểu về chương trình mới."
]

for i, test in enumerate(test_cases):
    result = preprocess(test)
    print(f"\n📝 Test {i+1}:")
    print(f"   Input: {test}")
    print(f"   Output: {result}")

print("\n" + "="*70)
print("✅ HÀM TIỀN XỬ LÝ ĐÃ SẴN SÀNG!")
print("="*70)

🔧 HÀM TIỀN XỬ LÝ VĂN BẢN TIẾNG VIỆT

🧪 TEST HÀM TIỀN XỬ LÝ:
--------------------------------------------------

📝 Test 1:
   Input: Hôm nay là một ngày đẹp trời, tôi đi học và gặp nhiều bạn bè!
   Output: ['hôm_nay', 'đẹp', 'tôi', 'h_c', 'gặp', 'bạn_bè']

📝 Test 2:
   Input: Thời tiết hôm nay thật tuyệt vời, trời xanh mây trắng.
   Output: ['i_tiết', 'hôm_nay', 'thật', 'tuy', 'xanh', 'mây', 'trắng']

📝 Test 3:
   Input: iPhone 15 Pro Max vừa được Apple ra mắt với nhiều tính năng mới.
   Output: ['iphone', 'pro', 'max', 'a_đư', 'apple', 'ra_mắt', 'tính', 'năng']

📝 Test 4:
   Input: Bộ trưởng Bộ Giáo dục và Đào tạo phát biểu về chương trình mới.
   Output: ['trư', 'b_giáo', 'đào_tạo', 'phát_biểu', 'chương_trình']

✅ HÀM TIỀN XỬ LÝ ĐÃ SẴN SÀNG!


In [19]:
# CELL 7: TIỀN XỬ LÝ - TỐI ƯU CHO 100K+ DOCUMENTS
print("="*70)
print("🔧 TIỀN XỬ LÝ DỮ LIỆU - TỐI ƯU TỐC ĐỘ")
print("="*70)

import time
import gc
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

processed_docs = []
start_time = time.time()

print(f"Đang xử lý {len(sample_docs):,} documents...")
print("(có thể mất 5-10 phút)")

# Xử lý theo batch để tiết kiệm RAM
BATCH_SIZE = 10000
total_batches = (len(sample_docs) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_idx in range(total_batches):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(sample_docs))
    batch_docs = sample_docs[start_idx:end_idx]
    
    batch_processed = []
    for doc in batch_docs:
        tokens = preprocess(doc)
        if tokens:
            batch_processed.append(tokens)
    
    processed_docs.extend(batch_processed)
    
    print(f"   ✅ Batch {batch_idx+1}/{total_batches}: {len(batch_processed):,} docs | Tổng: {len(processed_docs):,}")
    
    # Giải phóng bộ nhớ mỗi batch
    del batch_processed
    gc.collect()

elapsed = time.time() - start_time

print(f"\n✅ KẾT QUẢ TIỀN XỬ LÝ:")
print(f"   - Documents ban đầu: {len(sample_docs):,}")
print(f"   - Documents sau xử lý: {len(processed_docs):,}")
print(f"   - Tỷ lệ giữ lại: {len(processed_docs)/len(sample_docs)*100:.1f}%")
print(f"   - Thời gian: {elapsed/60:.2f} phút ({elapsed:.2f} giây)")

# Thống kê độ dài
if processed_docs:
    doc_lengths = [len(doc) for doc in processed_docs]
    print(f"\n📏 THỐNG KÊ ĐỘ DÀI:")
    print(f"   - Trung bình: {sum(doc_lengths)/len(doc_lengths):.2f} từ")
    print(f"   - Ngắn nhất: {min(doc_lengths)} từ")
    print(f"   - Dài nhất: {max(doc_lengths)} từ")

# Giải phóng
gc.collect()
print(f"\n💾 RAM sau tiền xử lý: ~{len(processed_docs) * 0.0001:.1f} GB")

🔧 TIỀN XỬ LÝ DỮ LIỆU - TỐI ƯU TỐC ĐỘ
Đang xử lý 150,000 documents...
(có thể mất 5-10 phút)
   ✅ Batch 1/15: 10,000 docs | Tổng: 10,000
   ✅ Batch 2/15: 10,000 docs | Tổng: 20,000
   ✅ Batch 3/15: 9,998 docs | Tổng: 29,998
   ✅ Batch 4/15: 10,000 docs | Tổng: 39,998
   ✅ Batch 5/15: 10,000 docs | Tổng: 49,998
   ✅ Batch 6/15: 9,996 docs | Tổng: 59,994
   ✅ Batch 7/15: 9,998 docs | Tổng: 69,992
   ✅ Batch 8/15: 9,999 docs | Tổng: 79,991
   ✅ Batch 9/15: 9,999 docs | Tổng: 89,990
   ✅ Batch 10/15: 10,000 docs | Tổng: 99,990
   ✅ Batch 11/15: 9,998 docs | Tổng: 109,988
   ✅ Batch 12/15: 9,999 docs | Tổng: 119,987
   ✅ Batch 13/15: 9,997 docs | Tổng: 129,984
   ✅ Batch 14/15: 9,999 docs | Tổng: 139,983
   ✅ Batch 15/15: 9,998 docs | Tổng: 149,981

✅ KẾT QUẢ TIỀN XỬ LÝ:
   - Documents ban đầu: 150,000
   - Documents sau xử lý: 149,981
   - Tỷ lệ giữ lại: 100.0%
   - Thời gian: 166.86 phút (10011.51 giây)

📏 THỐNG KÊ ĐỘ DÀI:
   - Trung bình: 392.05 từ
   - Ngắn nhất: 1 từ
   - Dài nhất: 2372

In [21]:
# CELL 8: CÁCH ĐƠN GIẢN - KHÔNG GIỚI HẠN SỐ TỪ
print("="*70)
print("📚 TẠO DICTIONARY VÀ CORPUS")
print("="*70)

from gensim import corpora
import gc

# Tạo dictionary
print("Đang tạo dictionary...")
dictionary = corpora.Dictionary(processed_docs)
print(f"✅ Dictionary ban đầu: {len(dictionary):,} từ")

# Chỉ lọc từ quá hiếm và quá phổ biến
print("\nĐang lọc từ...")
dictionary.filter_extremes(no_below=10, no_above=0.6)
print(f"✅ Sau khi lọc: {len(dictionary):,} từ")

# KHÔNG giới hạn số từ (để nguyên)

# Tạo corpus
print("\nĐang tạo corpus...")
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
print(f"✅ Corpus: {len(corpus):,} documents")

# Thống kê
print(f"\n📊 THỐNG KÊ:")
print(f"   - Từ vựng: {len(dictionary):,}")
print(f"   - Documents: {len(corpus):,}")

# Hiển thị mẫu từ vựng
print(f"\n📝 MẪU TỪ VỰNG (30 từ đầu tiên):")
for i, word in enumerate(list(dictionary.token2id.keys())[:30]):
    freq = dictionary.dfs.get(dictionary.token2id[word], 0)
    print(f"   {i+1:2d}. {word:20s} (xuất hiện: {freq:,} lần)")

# Giải phóng
gc.collect()

📚 TẠO DICTIONARY VÀ CORPUS
Đang tạo dictionary...
✅ Dictionary ban đầu: 1,405,581 từ

Đang lọc từ...
✅ Sau khi lọc: 96,905 từ

Đang tạo corpus...
✅ Corpus: 149,981 documents

📊 THỐNG KÊ:
   - Từ vựng: 96,905
   - Documents: 149,981

📝 MẪU TỪ VỰNG (30 từ đầu tiên):
    1. a_ch                 (xuất hiện: 42,481 lần)
    2. a_giải               (xuất hiện: 1,161 lần)
    3. accept               (xuất hiện: 85 lần)
    4. access               (xuất hiện: 360 lần)
    5. agreement            (xuất hiện: 61 lần)
    6. bao                  (xuất hiện: 37,076 lần)
    7. bao_hàm              (xuất hiện: 464 lần)
    8. biến_hóa             (xuất hiện: 743 lần)
    9. bung                 (xuất hiện: 840 lần)
   10. bên                  (xuất hiện: 54,760 lần)
   11. bạn                  (xuất hiện: 65,062 lần)
   12. bản                  (xuất hiện: 20,922 lần)
   13. bảng                 (xuất hiện: 13,982 lần)
   14. bảng_tính            (xuất hiện: 60 lần)
   15. c_lưu                (xuấ

0

In [22]:
# CELL 9: TRAINING LDA - CẤU HÌNH CAO CHO 100K+ DOCUMENTS
print("="*70)
print("🤖 TRAINING LDA - CẤU HÌNH CAO")
print("="*70)

from gensim import models
import time
import gc

# ============ CẤU HÌNH THEO SỐ LƯỢNG DOCUMENTS ============
if len(corpus) >= 100000:
    NUM_TOPICS = 30
    PASSES = 12
    ITERATIONS = 150
    CHUNKSIZE = 20000
    print("📊 CẤU HÌNH CAO (100,000+ documents)")
elif len(corpus) >= 50000:
    NUM_TOPICS = 25
    PASSES = 10
    ITERATIONS = 120
    CHUNKSIZE = 15000
    print("📊 CẤU HÌNH TRUNG BÌNH CAO (50,000-100,000 documents)")
else:
    NUM_TOPICS = 20
    PASSES = 8
    ITERATIONS = 100
    CHUNKSIZE = 10000
    print("📊 CẤU HÌNH TRUNG BÌNH (<50,000 documents)")

print(f"\n📊 CẤU HÌNH CHI TIẾT:")
print(f"   - Số topics: {NUM_TOPICS}")
print(f"   - Số passes: {PASSES}")
print(f"   - Số iterations: {ITERATIONS}")
print(f"   - Chunksize: {CHUNKSIZE}")
print(f"   - Số documents: {len(corpus):,}")
print(f"   - Từ vựng: {len(dictionary):,}")

# Ước lượng
estimated_ram = (len(corpus) * len(dictionary) * NUM_TOPICS * 8) / (1024**3)
print(f"💾 RAM ước tính cần: ~{estimated_ram:.2f} GB")
print(f"⏱️ Thời gian dự kiến: 15-25 phút")

print("\n⏳ ĐANG TRAINING...")
start_time = time.time()

lda_model = models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=PASSES,
    iterations=ITERATIONS,
    chunksize=CHUNKSIZE,
    alpha='auto',
    eta='auto',
    per_word_topics=True,
    update_every=1  # Cập nhật thường xuyên hơn
)

elapsed = time.time() - start_time
print(f"\n✅ TRAINING HOÀN TẤT!")
print(f"⏱️ Thời gian: {elapsed/60:.2f} phút ({elapsed:.2f} giây)")

# Giải phóng
gc.collect()

🤖 TRAINING LDA - CẤU HÌNH CAO
📊 CẤU HÌNH CAO (100,000+ documents)

📊 CẤU HÌNH CHI TIẾT:
   - Số topics: 30
   - Số passes: 12
   - Số iterations: 150
   - Chunksize: 20000
   - Số documents: 149,981
   - Từ vựng: 96,905
💾 RAM ước tính cần: ~3248.58 GB
⏱️ Thời gian dự kiến: 15-25 phút

⏳ ĐANG TRAINING...

✅ TRAINING HOÀN TẤT!
⏱️ Thời gian: 22.97 phút (1378.45 giây)


0

In [23]:
# CELL 10: HIỂN THỊ TOPICS
print("="*70)
print("📚 CÁC CHỦ ĐỀ PHÁT HIỆN")
print("="*70)

for idx in range(NUM_TOPICS):
    words = [w for w, _ in lda_model.show_topic(idx, topn=15)]
    print(f"\n📌 TOPIC {idx + 1}:")
    print(f"   {' → '.join(words)}")

📚 CÁC CHỦ ĐỀ PHÁT HIỆN

📌 TOPIC 1:
   trung_qu → nga → tàu → quân → tên → máy_bay → rằng → biển → khu → có_thể → hai → tấn_công → chính → đông → c_lư

📌 TOPIC 2:
   dân → sông → bắc → đất → núi → con → vùng → phía → đông → nhà → khu → mùa → nơi → cây → thành

📌 TOPIC 3:
   vi_t → nam → nguy_n → ông → dân → bài → viết → thành → sách → thế → văn_hóa → chính → tư_ng → hai → tên

📌 TOPIC 4:
   màu → thiết_kế → bạn → đẹp → nhà → phòng → chiếc → mẫu → mang → không_gian → tư_ng → có_thể → tạo → nhất → màu_sắc

📌 TOPIC 5:
   du_l → khách → nhất → biển → thành → khách_sạn → khu → nơi → du_khách → bạn → chuyến → đẹp → điểm → có_thể → bay

📌 TOPIC 6:
   bạn → có_thể → dùng → màn_hình → cách → n_thoại → phần_mềm → máy_tính → iphone → nhất → khác → máy → game → thông_tin → xem

📌 TOPIC 7:
   http → com → thép → cua → xem → nam → thu_mua → phế_li_u → xin → viet → ban → nhung → tin → https → www

📌 TOPIC 8:
   món → bạn → loại → ngon → rư_u → cách → cà_phê → nấu → hạt → thêm → rau → trà → dùng → bánh

In [24]:
# CELL 11: COHERENCE SCORE
print("="*70)
print("📊 COHERENCE SCORE")
print("="*70)

coherence_model = CoherenceModel(
    model=lda_model,
    texts=processed_docs,
    dictionary=dictionary,
    coherence='c_v'
)

score = coherence_model.get_coherence()
print(f"✅ Coherence Score: {score:.4f}")

# Đánh giá
print("\n💡 ĐÁNH GIÁ:")
if score > 0.6:
    print("   ✅ RẤT TỐT - Topics rõ ràng, có ý nghĩa thực tế")
elif score > 0.5:
    print("   👍 TỐT - Topics khá rõ ràng")
elif score > 0.4:
    print("   ⚠️ TRUNG BÌNH - Có thể cần điều chỉnh tham số")
else:
    print("   ❌ THẤP - Cần tăng số lượng dữ liệu hoặc điều chỉnh preprocessing")

📊 COHERENCE SCORE
✅ Coherence Score: 0.5237

💡 ĐÁNH GIÁ:
   👍 TỐT - Topics khá rõ ràng


In [25]:
# CELL 12: LƯU MÔ HÌNH
print("="*70)
print("💾 LƯU MÔ HÌNH")
print("="*70)

lda_model.save('/kaggle/working/lda_model.gensim')
dictionary.save('/kaggle/working/dictionary.gensim')

# Lưu metadata
metadata = {
    'num_topics': NUM_TOPICS,
    'passes': PASSES,
    'iterations': ITERATIONS,
    'vocabulary_size': len(dictionary),
    'num_documents': len(corpus),
    'coherence_score': score,
    'total_original_docs': total_rows,
    'sample_size': len(sample_docs)
}

with open('/kaggle/working/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("✅ ĐÃ LƯU:")
print("   - /kaggle/working/lda_model.gensim")
print("   - /kaggle/working/dictionary.gensim")
print("   - /kaggle/working/metadata.json")

print("\n" + "="*70)
print("🎉 HOÀN TẤT! MÔ HÌNH ĐÃ SẴN SÀNG")
print("="*70)

💾 LƯU MÔ HÌNH
✅ ĐÃ LƯU:
   - /kaggle/working/lda_model.gensim
   - /kaggle/working/dictionary.gensim
   - /kaggle/working/metadata.json

🎉 HOÀN TẤT! MÔ HÌNH ĐÃ SẴN SÀNG


In [26]:
# CELL: KIỂM TRA THÔNG TIN MÔ HÌNH
print("="*70)
print("📊 THÔNG TIN MÔ HÌNH LDA")
print("="*70)

import json
from gensim import models, corpora

# Đọc metadata
with open('/kaggle/working/metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(f"\n📈 THÔNG SỐ MÔ HÌNH:")
for key, value in metadata.items():
    if key != 'processed_docs':  # Bỏ qua processed_docs nếu có
        print(f"   {key}: {value}")

# Load model để kiểm tra
lda_model = models.LdaModel.load('/kaggle/working/lda_model.gensim')
dictionary = corpora.Dictionary.load('/kaggle/working/dictionary.gensim')

print(f"\n✅ Model loaded successfully!")
print(f"   - Số topics: {lda_model.num_topics}")
print(f"   - Dictionary size: {len(dictionary)}")

📊 THÔNG TIN MÔ HÌNH LDA

📈 THÔNG SỐ MÔ HÌNH:
   num_topics: 30
   passes: 12
   iterations: 150
   vocabulary_size: 96905
   num_documents: 149981
   coherence_score: 0.5237112226453302
   total_original_docs: 160000
   sample_size: 150000

✅ Model loaded successfully!
   - Số topics: 30
   - Dictionary size: 96905


In [28]:
# CELL: TẠO HÀM DỰ ĐOÁN TOPIC
print("="*70)
print("🔮 TẠO HÀM DỰ ĐOÁN TOPIC")
print("="*70)

from gensim import models, corpora
import re
from underthesea import word_tokenize

# Load model
lda_model = models.LdaModel.load('/kaggle/working/lda_model.gensim')
dictionary = corpora.Dictionary.load('/kaggle/working/dictionary.gensim')

# Stopwords (giống với khi training)
STOPWORDS = {
    'và', 'của', 'một', 'những', 'cho', 'với', 'các', 'đã', 'sẽ', 'có', 'còn',
    'được', 'không', 'này', 'nên', 'như', 'khi', 'vào', 'ra', 'thì', 'mà', 'rất',
    'để', 'cũng', 'nếu', 'vì', 'tại', 'hơn', 'nữa', 'lại', 'thấy', 'biết',
    'đó', 'là', 'từ', 'trong', 'ngoài', 'trên', 'dưới', 'trước', 'sau', 'nay'
}

def preprocess_text(text):
    """Tiền xử lý văn bản"""
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'[^a-zA-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠàáâãèéêìíòóôõùúăđĩũơƯĂẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼỀỀỂẾưăạảấầẩẫậắằẳẵặẹẻẽềềểế\s]', ' ', text)
    try:
        tokens = word_tokenize(text, format="text").split()
    except:
        tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

def predict_topic(text, top_n=3):
    """Dự đoán chủ đề cho văn bản"""
    tokens = preprocess_text(text)
    if not tokens:
        return []
    bow = dictionary.doc2bow(tokens)
    topics = lda_model.get_document_topics(bow)
    topics = sorted(topics, key=lambda x: x[1], reverse=True)
    return topics[:top_n]

def get_topic_words(topic_id, top_n=10):
    """Lấy top từ khóa của một topic"""
    return [w for w, _ in lda_model.show_topic(topic_id, topn=top_n)]

# Test
print("\n🧪 TEST HÀM DỰ ĐOÁN:")
test_texts = [
    "Đội tuyển Việt Nam giành chiến thắng trước Thái Lan",
    "Giá vàng hôm nay tăng mạnh, chạm mốc kỷ lục",
    "Công nghệ AI đang phát triển rất nhanh trong thời gian gần đây"
]

for text in test_texts:
    print(f"\n📝 Input: {text}")
    topics = predict_topic(text)
    print(f"   Kết quả:")
    for topic_id, prob in topics:
        words = get_topic_words(topic_id, 5)
        print(f"      Topic {topic_id+1} ({prob:.2%}): {', '.join(words)}")

🔮 TẠO HÀM DỰ ĐOÁN TOPIC

🧪 TEST HÀM DỰ ĐOÁN:

📝 Input: Đội tuyển Việt Nam giành chiến thắng trước Thái Lan
   Kết quả:
      Topic 10 (78.17%): trận, đấu, bóng, cầu, giải
      Topic 11 (1.78%): bạn, có_thể, cách, phải, vi_c
      Topic 12 (1.37%): tôi, anh, con, mình, nói

📝 Input: Giá vàng hôm nay tăng mạnh, chạm mốc kỷ lục
   Kết quả:
      Topic 24 (81.35%): tiền, giá, tăng, ngân_hàng, giao
      Topic 11 (1.54%): bạn, có_thể, cách, phải, vi_c
      Topic 12 (1.19%): tôi, anh, con, mình, nói

📝 Input: Công nghệ AI đang phát triển rất nhanh trong thời gian gần đây
   Kết quả:
      Topic 30 (81.58%): nam, vi_t, phát_triển, kinh_tế, sản_xuất
      Topic 11 (1.54%): bạn, có_thể, cách, phải, vi_c
      Topic 12 (1.19%): tôi, anh, con, mình, nói


In [29]:
# CELL: LƯU MODEL VÀ HÀM DỰ ĐOÁN
print("="*70)
print("💾 LƯU TOÀN BỘ MODEL VÀ FUNCTIONS")
print("="*70)

import pickle
import os

# Lưu hàm preprocess
import inspect
preprocess_code = inspect.getsource(preprocess_text)

# Lưu stopwords
with open('/kaggle/working/stopwords.txt', 'w', encoding='utf-8') as f:
    for word in sorted(STOPWORDS):
        f.write(word + '\n')

# Tạo file sử dụng mô hình
usage_code = '''
# ============ CÁCH SỬ DỤNG MÔ HÌNH LDA ============
from gensim import models, corpora
import re
from underthesea import word_tokenize

# Load model
lda_model = models.LdaModel.load('lda_model.gensim')
dictionary = corpora.Dictionary.load('dictionary.gensim')

# Stopwords (đã dùng khi training)
STOPWORDS = set([line.strip() for line in open('stopwords.txt', 'r', encoding='utf-8')])

def preprocess_text(text):
    """Tiền xử lý văn bản"""
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'[^a-zA-ZÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĐĨŨƠàáâãèéêìíòóôõùúăđĩũơƯĂẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼỀỀỂẾưăạảấầẩẫậắằẳẵặẹẻẽềềểế\s]', ' ', text)
    try:
        tokens = word_tokenize(text, format="text").split()
    except:
        tokens = text.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]

def predict_topic(text, top_n=3):
    """Dự đoán chủ đề cho văn bản"""
    tokens = preprocess_text(text)
    if not tokens:
        return []
    bow = dictionary.doc2bow(tokens)
    topics = lda_model.get_document_topics(bow)
    return sorted(topics, key=lambda x: x[1], reverse=True)[:top_n]

# Ví dụ sử dụng
if __name__ == "__main__":
    text = "Đội tuyển Việt Nam giành chiến thắng"
    topics = predict_topic(text)
    print(f"Text: {text}")
    for topic_id, prob in topics:
        print(f"  Topic {topic_id+1}: {prob:.2%}")
'''

with open('/kaggle/working/usage_example.py', 'w', encoding='utf-8') as f:
    f.write(usage_code)

print("✅ Đã lưu các file:")
print("   - lda_model.gensim")
print("   - dictionary.gensim")
print("   - metadata.json")
print("   - stopwords.txt")
print("   - usage_example.py")

💾 LƯU TOÀN BỘ MODEL VÀ FUNCTIONS
✅ Đã lưu các file:
   - lda_model.gensim
   - dictionary.gensim
   - metadata.json
   - stopwords.txt
   - usage_example.py


In [30]:
# CELL: TẠO FILE ZIP ĐỂ TẢI VỀ
print("="*70)
print("📦 TẠO FILE ZIP ĐỂ DOWNLOAD")
print("="*70)

import zipfile
import os

# Các file cần nén
files_to_zip = [
    '/kaggle/working/lda_model.gensim',
    '/kaggle/working/dictionary.gensim',
    '/kaggle/working/metadata.json',
    '/kaggle/working/stopwords.txt',
    '/kaggle/working/usage_example.py'
]

zip_path = '/kaggle/working/lda_model_complete.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in files_to_zip:
        if os.path.exists(file_path):
            arcname = os.path.basename(file_path)
            zipf.write(file_path, arcname)
            print(f"   Added: {arcname}")
        else:
            print(f"   ❌ Not found: {file_path}")

# Kiểm tra kích thước
size_mb = os.path.getsize(zip_path) / (1024**2)
print(f"\n✅ Đã tạo: {zip_path}")
print(f"📦 Kích thước: {size_mb:.2f} MB")

# Hiển thị nội dung zip
print("\n📋 Nội dung file zip:")
with zipfile.ZipFile(zip_path, 'r') as zipf:
    for info in zipf.infolist():
        print(f"   - {info.filename} ({info.file_size / 1024:.2f} KB)")

📦 TẠO FILE ZIP ĐỂ DOWNLOAD
   Added: lda_model.gensim
   Added: dictionary.gensim
   Added: metadata.json
   Added: stopwords.txt
   Added: usage_example.py

✅ Đã tạo: /kaggle/working/lda_model_complete.zip
📦 Kích thước: 1.90 MB

📋 Nội dung file zip:
   - lda_model.gensim (385.06 KB)
   - dictionary.gensim (3318.15 KB)
   - metadata.json (0.21 KB)
   - stopwords.txt (0.23 KB)
   - usage_example.py (1.50 KB)
